# 02 - Data Cleaning & Derived Variables (Làm sạch dữ liệu & Tạo biến phái sinh)

**Mục đích:** 
- Hiện thực hóa các quyết định xử lý dữ liệu đã quan sát ở Notebook 01.
- Tạo các biến phái sinh quan trọng: `IsCancelled`, `Revenue`, `InvoiceValue`.
- Xuất 2 bảng dữ liệu sạch vào `data/processed/` phục vụ cho các Phase phân tích tiếp theo:
  1. `retail_cleaned.csv` (cấp độ dòng giao dịch)
  2. `invoice_data.csv` (cấp độ hóa đơn)

## 0. Chuẩn bị thư viện và nạp module

In [1]:
import sys
from pathlib import Path
import pandas as pd

# Tự động nhận diện thư mục gốc của project
current_dir = Path.cwd().resolve()
PROJECT_ROOT = current_dir.parent if current_dir.name == "notebooks" else current_dir
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.data.load_data import load_raw_data
from src.data.cleaning import clean_retail_data, create_invoice_level_data, save_processed_data

# Hiển thị số thực với 2 chữ số thập phân
pd.options.display.float_format = "{:.2f}".format

## 1. Nạp dữ liệu thô (Raw Data)

In [2]:
raw_df = load_raw_data(r"C:\Users\chaud\Downloads\online-retail-customer-analytics-main\online-retail-customer-analytics-main\data\raw\Online Retail .xlsx")
print("Kích thước dữ liệu gốc:", raw_df.shape)
raw_df.head()

Kích thước dữ liệu gốc: (541909, 8)


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.00,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.00,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.00,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.00,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.00,United Kingdom


## 2. Tiến hành làm sạch dữ liệu giao dịch

Quy trình xử lý theo đúng nguyên tắc nghiệp vụ:
1. **Xóa dòng trùng hoàn toàn (drop_duplicates):** Loại bỏ lỗi ghi nhận đúp.
2. **Tạo biến `IsCancelled`:** Đơn hủy bắt đầu bằng chữ cái `C`. Không xóa đơn hủy mà gắn cờ để kiểm định ở Phase 6.
3. **Lọc bỏ giao dịch bất thường:** Lọc bỏ `UnitPrice <= 0` (hàng tặng/nợ xấu) và `Quantity < 0` không phải đơn hủy (kiểm kê kho).
4. **Tạo biến doanh thu:** `Revenue = Quantity * UnitPrice`.

In [3]:
df_clean = clean_retail_data(raw_df)
print("\nKích thước sau làm sạch:", df_clean.shape)
df_clean.head()

--- Bắt đầu quy trình làm sạch dữ liệu giao dịch ---
Số dòng ban đầu: 541,909
1. Đã xóa dòng trùng lặp: 5,268 dòng
2. Gắn cờ IsCancelled: 9,251 dòng đơn hủy
3. Đã loại bỏ dòng bất thường (giá <= 0, điều chỉnh kho): 2,512 dòng
4. Đã tạo biến doanh thu: Revenue = Quantity * UnitPrice
-> Số dòng sau khi làm sạch: 534,129 (giữ lại 98.56%)

Kích thước sau làm sạch: (534129, 10)


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,IsCancelled,Revenue
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.00,United Kingdom,False,15.30
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.00,United Kingdom,False,20.34
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.00,United Kingdom,False,22.00
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.00,United Kingdom,False,20.34
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.00,United Kingdom,False,20.34


## 3. Kiểm tra tính toàn vẹn của dữ liệu sau làm sạch

In [4]:
print("--- Phân bố đơn hủy (IsCancelled) ---")
print(df_clean["IsCancelled"].value_counts())
print()

print("--- Kiểm tra giá trị Doanh thu (Revenue) ---")
print("Số dòng doanh thu âm (đơn hủy hoàn tiền):", (df_clean["Revenue"] < 0).sum())
print("Số dòng doanh thu dương (bán hàng thành công):", (df_clean["Revenue"] > 0).sum())
print()

# Thống kê mô tả 3 biến số lượng, đơn giá và doanh thu
df_clean[["Quantity", "UnitPrice", "Revenue"]].describe()

--- Phân bố đơn hủy (IsCancelled) ---
IsCancelled
False    524878
True       9251
Name: count, dtype: int64

--- Kiểm tra giá trị Doanh thu (Revenue) ---
Số dòng doanh thu âm (đơn hủy hoàn tiền): 9251
Số dòng doanh thu dương (bán hàng thành công): 524878



,Quantity,UnitPrice,Revenue
count,534129.00,534129.00,534129.00
mean,9.92,4.70,18.25
std,216.45,95.08,380.95
min,-80995.00,0.00,-168469.60
25%,1.00,1.25,3.75
50%,3.00,2.10,9.90
75%,10.00,4.13,17.57
max,80995.00,38970.00,168469.60


## 4. Tổng hợp dữ liệu cấp độ Hóa đơn (Invoice Level)

> **Phân biệt rõ:**
> - `Revenue`: Giá trị bán hàng ở mức từng dòng sản phẩm.
> - `InvoiceValue`: Tổng giá trị tiền tệ của một hóa đơn (`SUM(Revenue)`).

In [5]:
df_invoice = create_invoice_level_data(df_clean)
print("Kích thước bảng hóa đơn:", df_invoice.shape)
df_invoice.head()


--- Bắt đầu tổng hợp dữ liệu cấp độ Hóa đơn (Invoice level) ---
-> Tổng số hóa đơn duy nhất: 23,796
   - Hóa đơn mua hàng hợp lệ: 19,960
   - Hóa đơn hủy (cancellation): 3,836
Kích thước bảng hóa đơn: (23796, 8)


,InvoiceNo,InvoiceDate,CustomerID,Country,IsCancelled,InvoiceValue,TotalQuantity,ItemCount
0,536365,2010-12-01 08:26:00,17850.00,United Kingdom,False,139.12,40,7
1,536366,2010-12-01 08:28:00,17850.00,United Kingdom,False,22.20,12,2
2,536367,2010-12-01 08:34:00,13047.00,United Kingdom,False,278.73,83,12
3,536368,2010-12-01 08:34:00,13047.00,United Kingdom,False,70.05,15,4
4,536369,2010-12-01 08:35:00,13047.00,United Kingdom,False,17.85,3,1


## 5. Lưu kết quả ra thư mục `data/processed/`

In [6]:
save_processed_data(df_clean, df_invoice)
print("Lưu file hoàn tất!")


--- Đang lưu dữ liệu đã xử lý vào: C:\Users\chaud\Downloads\online-retail-customer-analytics-main\online-retail-customer-analytics-main\data\processed ---
-> Đã lưu: retail_cleaned.csv (52.43 MB)
-> Đã lưu: invoice_data.csv (1.60 MB)
Lưu file hoàn tất!
